# Importações e verificações

In [1]:
# %%capture --no-stderr
# %pip install -U langchain langchain_openai langsmith pandas langchain_experimental matplotlib langgraph langchain_core langchain_anthropic

In [2]:
import getpass
import os
import operator
import functools
from typing import Annotated, Sequence, TypedDict, Literal

from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    ToolMessage,
    AIMessage,
)
from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_experimental.utilities import PythonREPL
from langchain_openai import ChatOpenAI

from langgraph.graph import END, StateGraph, START
from langgraph.prebuilt import ToolNode

In [3]:
def _set_if_undefined(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"Por favor, forneça seu  {var}")


_set_if_undefined("TAVILY_API_KEY")
_set_if_undefined("ANTHROPIC_API_KEY")
_set_if_undefined("OPENAI_API_KEY")
_set_if_undefined("LANGCHAIN_API_KEY")

# Optional, add tracing in LangSmith
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Multi-agent Collaboration"

Por favor, forneça seu  TAVILY_API_KEY··········
Por favor, forneça seu  ANTHROPIC_API_KEY··········
Por favor, forneça seu  OPENAI_API_KEY··········
Por favor, forneça seu  LANGCHAIN_API_KEY··········


# Criar agentes

As funções auxiliares a seguir ajudarão a criar agentes. Esses agentes serão nós no gráfico.

Você pode pular adiante se quiser apenas ver como é o gráfico.

In [4]:
def create_agent(llm, tools, system_message: str):
    """Create an agent."""
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "Você é um assistente de IA útil, colaborando com outros assistentes."
                "Use as ferramentas fornecidas para avançar na resposta à pergunta."
                "Se você não conseguir responder totalmente, tudo bem, outro assistente com ferramentas diferentes "
                "ajudará de onde você parou. Execute o que puder para progredir."
                "Se você ou qualquer outro assistente tiver a resposta ou resultado final,"
                "prefixe sua resposta com RESPOSTA FINAL para que a equipe saiba que deve parar."
                "Você tem acesso às seguintes ferramentas: {tool_names}.\n{system_message}",
            ),
            MessagesPlaceholder(variable_name="messages"),
        ]
    )
    prompt = prompt.partial(system_message=system_message)
    prompt = prompt.partial(tool_names=", ".join([tool.name for tool in tools]))
    return prompt | llm.bind_tools(tools)

# Definir ferramentas


Também definiremos algumas ferramentas que nossos agentes usarão no futuro

In [5]:
tavily_tool = TavilySearchResults(max_results=5)

# Aviso: isso executa código localmente,
# o que pode ser inseguro quando não estiver em sandbox

repl = PythonREPL()

@tool
def python_repl(
    code: Annotated[str, "The python code to execute to generate your chart."],
):
    """Use isso para executar o código python. Se você quiser ver a saída de um valor,
    você deve imprimi-lo com `print(...)`. Isso é visível para o usuário."""
    try:
        result = repl.run(code)
    except BaseException as e:
        return f"Falha ao executar. Error: {repr(e)}"
    result_str = f"Executado com sucesso:\n```python\n{code}\n```\nStdout: {result}"
    return (
        result_str + "\n\nSe você concluiu todas as tarefas, responda com RESPOSTA FINAL."
    )


# Definir estado


Primeiro, definimos o estado do gráfico. Isso será apenas uma lista de mensagens, juntamente com uma chave para rastrear o remetente mais recente

In [6]:
# Isso define o objeto que é passado entre cada nó
# no gráfico. Criaremos nós diferentes para cada agente e ferramenta
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    sender: str

# Definir nós de agente

Agora precisamos definir os nós. Primeiro, vamos definir os nós para os agentes.

In [7]:
# Função auxiliar para criar um nó para um determinado agente
def agent_node(state, agent, name):
    result = agent.invoke(state)
    # Convertemos a saída do agente em um formato adequado para anexar ao estado global
    if isinstance(result, ToolMessage):
        pass
    else:
        result = AIMessage(**result.dict(exclude={"type", "name"}), name=name)
    return {
        "messages": [result],
        # Como temos um fluxo de trabalho rigoroso, podemos
        # rastrear o remetente para sabermos para quem passar em seguida.
        "sender": name,
    }

In [8]:
# llm = ChatOpenAI(model="gpt-4-1106-preview")
llm = ChatAnthropic(model="claude-3-sonnet-20240229", temperature=1)

# Agente de pesquisa e nó
research_agent = create_agent(
    llm,
    [tavily_tool],
    system_message="Você deve fornecer dados precisos para o chart_generator usar.",
)
research_node = functools.partial(agent_node, agent=research_agent, name="Researcher")

# chart_generator
chart_agent = create_agent(
    llm,
    [python_repl],
    system_message="Todos os gráficos que você exibir ficarão visíveis para o usuário.",
)
chart_node = functools.partial(agent_node, agent=chart_agent, name="chart_generator")


# Definir nó da ferramenta

Agora definimos um nó para executar as ferramentas

In [9]:
tools = [tavily_tool, python_repl]
tool_node = ToolNode(tools)

# Definir a lógica de borda

Podemos definir parte da lógica de borda necessária para decidir o que fazer com base nos resultados dos agentes

In [10]:
def router(state) -> Literal["call_tool", "__end__", "continue"]:
    # Este é o roteador
    messages = state["messages"]
    last_message = messages[-1]
    if last_message.tool_calls:
        # O agente anterior está invocando uma ferramenta
        return "call_tool"
    if "FINAL ANSWER" in last_message.content:
        # Qualquer agente decidiu que o trabalho está feito
        return "__end__"
    return "continue"

# Definir o gráfico

Agora podemos juntar tudo e definir o gráfico!

In [11]:
workflow = StateGraph(AgentState)

workflow.add_node("Researcher", research_node)
workflow.add_node("chart_generator", chart_node)
workflow.add_node("call_tool", tool_node)

workflow.add_conditional_edges(
    "Researcher",
    router,
    {"continue": "chart_generator", "call_tool": "call_tool", "__end__": END},
)
workflow.add_conditional_edges(
    "chart_generator",
    router,
    {"continue": "Researcher", "call_tool": "call_tool", "__end__": END},
)

workflow.add_conditional_edges(
    "call_tool",
    # Cada nó do agente atualiza o campo 'remetente'
    # o nó de chamada da ferramenta não, o que significa
    # esta borda será roteada de volta para o agente original
    # quem invocou o também
    lambda x: x["sender"],
    {
        "Researcher": "Researcher",
        "chart_generator": "chart_generator",
    },
)
workflow.add_edge(START, "Researcher")
graph = workflow.compile()

In [ ]:
events = graph.stream(
    {
        "messages": [
            HumanMessage(
                content="Fetch the UK's GDP over the past 5 years,"
                " then draw a line graph of it."
                " Once you code it up, finish."
            )
        ],
    },
    # Maximum number of steps to take in the graph
    {"recursion_limit": 150},
)

for s in events:
    print(s)
    print("----")